In [ ]:
# 1. ROBUST DEPENDENCIES SETUP

import sys
import subprocess

print("Configuring dynamic accent evaluation engine... Please wait.")
subprocess.run(["apt-get", "update"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "libasound-dev", "portaudio19-dev", "libportaudio2", "libportaudiocpp0", "ffmpeg"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import speech_recognition as sr
    import pronouncing
    from rapidfuzz import fuzz
    from gtts import gTTS
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "speechrecognition", "pyaudio", "pronouncing", "rapidfuzz", "gTTS"])

import speech_recognition as sr
import pronouncing
import numpy as np
import wave
from rapidfuzz import fuzz
from gtts import gTTS
import base64
from google.colab import output
from IPython.display import HTML, display, Audio

# 2. DYNAMIC PHONEME GENERATOR (INFINITE WORD SUPPORT VIA RULES)

def get_accented_phonemes(word, language):
    word = word.lower().strip()

    # Get base American phonetics from CMU dictionary
    base_phones_list = pronouncing.phones_for_word(word)

    # Backup generator for new or strange words missing from the dictionary
    if not base_phones_list:
        # Project safty: Create a rough guess if the word is missing from the dictionary
        vowels = 'aeiouy'
        dummy_phones = []
        for char in word:
            if char in vowels: dummy_phones.append("AH0")
            elif char.isalpha(): dummy_phones.append(char.upper())
        return [" ".join(dummy_phones)]

    base_phones = base_phones_list[0]

    # If user selected American, use base phonetics

    if language == "en-US":
        return [base_phones]


    # Global Phonetic Rules for British Accent (Universal Rules)

    phones = base_phones.split()
    new_phones = []

    for i, p in enumerate(phones):
        p_clean = ''.join([c for c in p if not c.isdigit()])
        stress = ''.join([c for c in p if c.isdigit()])
        stress_str = stress if stress else '0'

        # Rule 1: Non-rhoticity (Delete 'R' at the end of words or right before a consonant)
        if p_clean == 'R':
            if i == len(phones) - 1:  # Delete 'R' if it is the last letter of the word
                continue
            elif i < len(phones) - 1 and ''.join([c for c in phones[i+1] if not c.isdigit()]) not in ['A','E','I','O','U','Y']:
                continue  # Delete 'R' if the next letter is not a vowel

        # Rule 2: Post-vocalic ER-to-AH Shift (Convert the final 'ER' -> 'AH' in words like water, better, and computer 'ER' -> 'AH')
        if p_clean == 'ER':
            new_phones.append(f"AH{stress_str}")
            continue

        # Rule 3: Broad 'A' / Bath Vowel Shift (In words such as fast, ask, dance, and path, convert AE → AA for the British accent.)
        # British English pronounces 'fast' as /fɑːst/ (AA), while American English pronounces it as /faest/ (AE)
        if p_clean == 'AE' and word in ["fast", "ask", "dance", "path", "class", "glass", "master", "after", "can't"]:
            new_phones.append(f"AA{stress_str}")
            continue

        # Rule 4: Tomato/Vase Yod & Vowel Shift
        if word == "tomato" and p_clean == 'EY':
            new_phones.append(f"AA{stress_str}")
            continue
        if word == "schedule" and p_clean == 'S' and i == 0:
            new_phones.append("SH")
            continue

        new_phones.append(p)

    return [" ".join(new_phones)]

def calculate_acoustic_accent_distance(user_wav, native_wav):
    try:
        with wave.open(user_wav, 'rb') as w1:
            sound1 = np.frombuffer(w1.readframes(-1), dtype=np.int16).astype(np.float32)
        with wave.open(native_wav, 'rb') as w2:
            sound2 = np.frombuffer(w2.readframes(-1), dtype=np.int16).astype(np.float32)

        if len(sound1) == 0 or len(sound2) == 0: return 0.3

        intervals = 50
        env1 = np.array([np.sqrt(np.mean(x**2)) for x in np.array_split(sound1, intervals)])
        env2 = np.array([np.sqrt(np.mean(x**2)) for x in np.array_split(sound2, intervals)])

        env1 /= (np.max(env1) if np.max(env1) > 0 else 1.0)
        env2 /= (np.max(env2) if np.max(env2) > 0 else 1.0)

        return np.mean(np.abs(env1 - env2))
    except:
        return 0.3


# 3. HIGH-SENSITIVITY COLAB AUDIO RECORDING INTERFACE

def record_voice_from_browser():
    js_interface = """
    async function record() {
      const stream = await navigator.mediaDevices.getUserMedia({
        audio: { echoCancellation: true, noiseSuppression: true, channelCount: 1, sampleRate: 16000 }
      });
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      const mainDiv = document.createElement('div');
      mainDiv.style.padding = '20px'; mainDiv.style.background = '#f8f9fa';
      mainDiv.style.borderRadius = '10px'; mainDiv.style.border = '2px dashed #e67e22';
      mainDiv.style.width = '350px'; mainDiv.style.textAlign = 'center'; mainDiv.style.margin = '15px 0';
      const statusText = document.createElement('p');
      statusText.textContent = 'Status: Ready to Practice'; statusText.style.fontWeight = 'bold';
      const btn = document.createElement('button');
      btn.textContent = '🎙️ Start Recording'; btn.style.background = '#e67e22';
      btn.style.color = 'white'; btn.style.padding = '12px 24px'; btn.style.border = 'none';
      btn.style.borderRadius = '25px'; btn.style.cursor = 'pointer'; btn.style.fontWeight = 'bold';
      mainDiv.appendChild(statusText); mainDiv.appendChild(btn); document.body.appendChild(mainDiv);
      recorder.ondataavailable = (e) => chunks.push(e.data);
      btn.onclick = () => {
        if (recorder.state === "inactive") {
          chunks.length = 0; recorder.start();
          statusText.textContent = 'Status: 🔴 Recording... Speak Clearly!';
          btn.textContent = '🛑 Stop & Analyze'; btn.style.background = '#e74c3c';
        } else {
          recorder.stop(); statusText.textContent = 'Status: Analyzing Accent...';
          btn.textContent = 'Analyzing...'; btn.disabled = true;
        }
      };
      return new Promise(resolve => {
        recorder.onstop = async () => {
          const blob = new Blob(chunks, { type: 'audio/webm' });
          const reader = new FileReader(); reader.readAsDataURL(blob);
          reader.onloadend = () => resolve(reader.result.split(',')[1]);
          stream.getTracks().forEach(track => track.stop());
          setTimeout(() => mainDiv.remove(), 1000);
        };
      });
    }
    """
    display(HTML(f"<script>{js_interface}</script>"))
    return base64.b64decode(output.eval_js("record()"))


# 4. MAIN INTERFACE (With Target Phoneme on Critical Error)

print("=== ADVANCED AI ACCENT ANALYZER ===")
print("\nSelect target accent environment:")
print("1. American English (en-US)")
print("2. British English (en-GB)")
accent_choice = input("Enter choice (1 or 2): ").strip()

if accent_choice == "2":
    selected_language, gtts_lang, tld_option, accent_label = "en-GB", "en", "co.uk", "British English (en-GB)"
else:
    selected_language, gtts_lang, tld_option, accent_label = "en-US", "en", "com", "American English (en-US)"

target_text = input("\nEnter the target word or sentence to practice: ").strip()
clean_target = target_text.translate(str.maketrans('', '', '?.!,-')).lower().strip()

print(f"\n[Environment Configured: {accent_label}]")
print("-> Click the button below, record your voice clearly, and press 'Stop'.")

try:
    raw_audio = record_voice_from_browser()
    with open("user_input.webm", "wb") as f: f.write(raw_audio)

    subprocess.run(["ffmpeg", "-y", "-i", "user_input.webm", "-ac", "1", "-ar", "16000", "user_audio.wav"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Native Reference Generation
    tts = gTTS(text=target_text, lang=gtts_lang, tld=tld_option, slow=False)
    tts.save("native_output.mp3")
    subprocess.run(["ffmpeg", "-y", "-i", "native_output.mp3", "-ac", "1", "-ar", "16000", "native_audio.wav"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 300
    recognizer.dynamic_energy_threshold = False

    with sr.AudioFile("user_audio.wav") as source:
        audio_content = recognizer.record(source)

    try:
        spoken_text = recognizer.recognize_google(audio_content, language=selected_language)
        clean_spoken = spoken_text.translate(str.maketrans('', '', '?.!,-')).lower().strip()

        print("\n" + "="*45)
        print(f"Target Text   : {target_text}")
        print(f"Your Delivery : \"{spoken_text}\"")
        print("="*45)

        # Sequence Validation
        strict_ratio = fuzz.ratio(clean_target, clean_spoken)
        token_sort_ratio = fuzz.token_sort_ratio(clean_target, clean_spoken)
        combined_text_score = (strict_ratio + token_sort_ratio) / 2

        if combined_text_score < 55:
            # 0% Dashboard Display for Completely Incorrect Sentences

            print(f"\n🎯 Actual Accent Match Score: 0%")
            print("-" * 45)
            print("❌ CRITICAL ERROR: COMPLETELY DIFFERENT SENTENCE DETECTED!")
            print("\n")
            print(f"• Expected : '{target_text}'")
            print(f"• Detected : '{spoken_text}'")
            print("• Solution : Please practice pronunciation accuracy and context.")
            print("\n")

            # Accurate Phonetic Guideline Generation for the Target Sentence

            print(f"📖 CORRECT PHONETIC GUIDANCE FOR '{target_text}':")
            print("\n")
            target_words = clean_target.split()
            for word in target_words:
                correct_phones = get_accented_phonemes(word, selected_language)
                print(f" • Word: '{word}' -> Correct Target Pattern: {correct_phones}")
            print("\n")

        else:
            # Acoustic Envelope Scoring (for accurate delivery)
            acoustic_diff = calculate_acoustic_accent_distance("user_audio.wav", "native_audio.wav")
            raw_accent_score = 100 - (acoustic_diff * 140)

            final_score = int(raw_accent_score * (combined_text_score / 100))
            final_score = max(15, min(98, final_score))

            print(f"\n🎯 Actual Accent Match Score: {final_score}%")
            print("-" * 45)

            target_words = clean_target.split()
            spoken_words = clean_spoken.split()

            missed_words = [word for word in target_words if word not in spoken_words]

            if missed_words or final_score < 78:
                print("❌ INCORRECT / MISSED WORDS DETECTED:")
                print("\n")
                words_to_report = missed_words if missed_words else target_words
                for word in words_to_report:
                    phones_list = get_accented_phonemes(word, selected_language)
                    print(f" • Word: '{word}' was spoken incorrectly. Pattern: {phones_list}")
                print("-" * 45)

            print("💡 ACCENT ASSESSMENT & SUGGESTIONS:")
            if selected_language == "en-US":
                print("- US Accent Fix: Emphasize the 'R' sounds (Rhoticity) and turn 'T' into a soft 'D' between vowels.")
            else:
                print("- UK Accent Fix: Drop the 'R' sound at the end of words (Non-rhotic) and make the 'T' sound very crisp.")

            if final_score >= 80:
                print("- Feedback: Excellent delivery pacing. Your speech envelopes match perfectly.")
            else:
                print("- Feedback: Regional cadence mismatch detected. Please mimic the native audio pacing below.")

        print(f"\n🎧 LISTEN TO THE CORRECT {accent_label.upper()} ACCENT:")
        display(Audio("native_output.mp3", autoplay=False))

    except sr.UnknownValueError:
        print("\nSystem Notice: Audio clarity too low to transcribe. Please speak closer to the mic.")
    except sr.RequestError:
        print("\nSystem Notice: Speech Recognition API communication error.")

except Exception as err:
    print(f"\nSystem Runtime Error: {err}")

Configuring dynamic accent evaluation engine... Please wait.
=== ADVANCED AI ACCENT ANALYZER ===

Select target accent environment:
1. American English (en-US)
2. British English (en-GB)
Enter choice (1 or 2): 1

Enter the target word or sentence to practice: red

[Environment Configured: American English (en-US)]
-> Click the button below, record your voice clearly, and press 'Stop'.



Target Text   : red
Your Delivery : "read"

🎯 Actual Accent Match Score: 58%
---------------------------------------------
❌ INCORRECT / MISSED WORDS DETECTED:


 • Word: 'red' was spoken incorrectly. Pattern: ['R EH1 D']
---------------------------------------------
💡 ACCENT ASSESSMENT & SUGGESTIONS:
- US Accent Fix: Emphasize the 'R' sounds (Rhoticity) and turn 'T' into a soft 'D' between vowels.
- Feedback: Regional cadence mismatch detected. Please mimic the native audio pacing below.

🎧 LISTEN TO THE CORRECT AMERICAN ENGLISH (EN-US) ACCENT:
